# v11: v10 navigation preserved + smoother steering and explicit finish detection

Same structure as v9 (NavigationCoordinator → GreenMarkerTurnController / GapCrossingController → BlackLineController,
runner is the only motor writer). Every change is tagged `# v10:` in the code. Nothing drives automatically. v11 adds light filtering during ordinary line following and red-then-silver finish detection; green-marker and gap logic are otherwise preserved. v11 adds light filtering during ordinary line following and a red-then-silver finish detector; green-marker and gap logic are otherwise preserved.

| Symptom on the course | Root cause found in v9 | v10 change | Embedded-design principle |
|---|---|---|---|
| Green marker ignored, or robot turns the wrong way at a junction | The crossbar of the junction is the *largest contour* in a band, so the band centroid jumps sideways. Steering error exceeds 0.55 → BlackLineController starts a TURN → coordinator blocks green (`can_start_green` false). The same shift corrupts the line direction used to classify the marker side. | Bands that contain a horizontal stroke report the **through-line** column instead of the centroid. Green may also pre-empt a black-line TURN when a junction is visible. | Sensor-data validation: reject a measurement that does not describe the quantity you control |
| Robot pivots before reaching the junction and loses the line | Fixed 0.15 s open-loop advance. When the marker triggers, the crossbar is still ~10–15 cm ahead of the wheel axle. | **Event-triggered advance**: drive (with near-band steering) until the crossbar leaves the bottom of the image, then a short calibrated *camera-blind* time. | Closed loop over open loop: an event from the sensor instead of a time that depends on battery voltage and floor friction |
| Second marker right after a turn is missed | Re-arming needs 5 consecutive frames with **no green anywhere** in the ROI. If the next marker is already visible, the controller never re-arms. | Markers count only if a crossbar is visible **above** them (rule 1.5.6: markers sit just before the junction). This spatial filter prevents re-triggering on an old marker, so re-arming needs only a short cooldown. | Replace a temporal guard with a spatial (physical) invariant from the specification |
| Marker confirmation flickers | 3 strictly consecutive frames; one UNKNOWN resets the count. | 3-of-5 vote; LEFT/RIGHT can upgrade to BOTH while stopped (late binding of the decision). | Debouncing / k-of-n filtering of a noisy digital input |
| U-turn stops at the side branch | Robot drives into the junction before turning 180°, so the crossbar looks like a centred line at 90°. | BOTH → turn around **before** the junction (no advance); minimum turn time derived from a measured rotation time. | Calibrated physical model instead of magic numbers |
| Tight bend → "turn timed out" / "line not reacquired" | Outer-wheel-only pivot moves the camera forward off the corner; lost-line limit 0.35 s; fixed 1.0 s turn limit regardless of progress. | Inner wheel reverses (rotation about the axle); timeouts derived from rotation time; a turn fails only if it stops making progress. | Bounded, progress-based watchdogs instead of fixed deadlines |
| Zig-zag bends cut | Look-ahead term (effective far weight 0.60) steers toward the *second* bend. | When near and far bands disagree strongly, steer on near/middle only and slow to the minimum reliable speed. | Graceful degradation |
| Hard to tell *why* a run failed | No record | Bounded ring-buffer log, written to CSV **after** the run; loop timing statistics. | No I/O in the control loop; bounded memory; measure, don't guess |

**Before floor tests:** run the synthetic self-test and a stationary preview. Verify the red/silver detector on the actual finish marker before driving unattended.

## 1. Hardware and camera

In [72]:
import time, csv
from collections import deque, namedtuple
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()
camera = Camera.instance(width=224, height=224)
live_view = widgets.Image(format="jpeg", width=336, height=336)
camera_link = traitlets.dlink((camera, "value"), (live_view, "value"), transform=bgr8_to_jpeg)
display(live_view)

RuntimeError: Could not initialize camera.  Please see error trace.

In [73]:
latest = {"sample": None}  # newest frame only: decisions are made on fresh data, never on a queue

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic())

camera.observe(receive_frame, names="value")
time.sleep(0.5)
print("Camera frame received." if latest["sample"] is not None else "No camera frame yet.")

Camera frame received.


## 2. Calibration constants (measure these on the real floor)
v10 collects every value that depends on the physical robot in one place, and derives timeouts from
measured quantities instead of tuning each timeout separately.

In [ ]:
import time, csv
from collections import deque, namedtuple
import cv2
import numpy as np

# Motor commands (copy your proven values)
BASE_SPEED = 0.12
CURVE_SPEED = 0.12
MIN_DRIVE_SPEED = 0.12    # v10 CALIBRATE: lowest command that still reliably moves the robot (avoid stalling)
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15         # black-line pivot, outer wheel
GREEN_TURN_SPEED = 0.15   # green turn, both wheels (one reversed)
PIVOT_INNER_RATIO = -0.6  # v10: inner wheel reverses at 60 % -> rotation near the axle. 0.0 = v9 behaviour

# Measured geometry/timing
ROTATION_TIME_360 = 2.15   # v10 CALIBRATE with spin_test(): seconds per full in-place turn at GREEN_TURN_SPEED
CAMERA_BLIND_TIME = 0.12  # v10 CALIBRATE: time at BASE_SPEED for tape leaving the image bottom to reach the axle
FRAME_TIMEOUT = 0.5

# Derived (re-run this cell after changing a calibration value)
QUARTER_TURN_TIME = ROTATION_TIME_360 / 4.0
# Black-line pivot rotates slower: wheel-speed difference is (1 - ratio)*TURN_SPEED instead of 2*GREEN_TURN_SPEED
LINE_PIVOT_TIME_360 = ROTATION_TIME_360 * (2.0 * GREEN_TURN_SPEED) / ((1.0 - PIVOT_INNER_RATIO) * TURN_SPEED)

def clamp(value, low, high):
    return max(low, min(high, value))

In [ ]:
def spin_test(seconds=2.15):
    """Spin in place for a fixed time; measure the angle turned and set ROTATION_TIME_360 = seconds*360/angle."""
    try:
        robot.set_motors(-GREEN_TURN_SPEED, GREEN_TURN_SPEED)
        time.sleep(seconds)
    finally:
        robot.stop()

spin_test(2.15)

## 3. Perception
Returns one `Observation` per frame. v10 adds: green pixels removed from the black mask, junction (crossbar)
detection, through-line position in junction bands, and the rule that a marker is actionable only when a
crossbar is visible above it.

In [ ]:
BLACK_THRESHOLD = 70
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

# v10: junction handling
CROSS_BAND_WIDTH = 0.40     # contour wider than this fraction of the image = horizontal stroke in the band
STROKE_FILL = 0.85          # a column black over this fraction of the band height belongs to the through-line
CROSS_ROW_FRACTION = 0.25   # a row with more black than this fraction of width belongs to a horizontal stroke
CROSS_MIN_ROWS = 3
CROSS_SEARCH = (0.30, 0.95)

GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0
GREEN_PAIR_Y_TOLERANCE = 0.15

# v11: finish marker is a red band followed by reflective silver foil.
FINISH_RED_MIN_COVERAGE = 0.30
FINISH_SILVER_MIN_COVERAGE = 0.48
FINISH_ROW_MIN_PIXELS = 3
FINISH_ROI_TOP = 0.42
FINISH_ROI_BOTTOM = 0.96
FINISH_SEQUENCE_TIMEOUT = 3.0

Observation = namedtuple("Observation", "positions direction junction_y zone_markers markers crossing finish")

def detect_green(hsv, height):
    colour = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    kernel = np.ones((3, 3), dtype=np.uint8)
    colour = cv2.morphologyEx(colour, cv2.MORPH_OPEN, kernel)
    colour = cv2.morphologyEx(colour, cv2.MORPH_CLOSE, kernel)
    roi = colour.copy()
    roi[:int(height * GREEN_ROI_TOP), :] = 0
    roi[int(height * GREEN_ROI_BOTTOM):, :] = 0
    contours = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0 or not 0.4 <= w / float(h) <= 2.5:
            continue
        m = cv2.moments(contour)
        if m["m00"] <= 0:
            continue
        markers.append(dict(x=m["m10"] / m["m00"], y=m["m01"] / m["m00"], area=area, box=(x, y, w, h)))
    markers.sort(key=lambda item: item["area"], reverse=True)
    return markers, colour, roi

def through_line_x(strip):
    """Find a through-line; preserve the original vertical-line check."""
    black = strip > 0
    height, width = black.shape

    # 1. Original v10 behavior.
    fill = black.mean(axis=0)
    cols = np.flatnonzero(fill >= STROKE_FILL)

    if cols.size:
        runs = np.split(
            cols, np.flatnonzero(np.diff(cols) > 1) + 1
        )
        return float(max(runs, key=len).mean())

    # 2. Additional check for a slightly slanted through-line.
    if height < 8:
        return None

    edge_rows = max(2, height // 8)
    max_line_width = 0.25 * width

    def edge_runs(edge):
        cols = np.flatnonzero(edge.mean(axis=0) >= 0.75)
        if cols.size == 0:
            return []

        runs = np.split(
            cols, np.flatnonzero(np.diff(cols) > 1) + 1
        )

        # A broad horizontal branch is not a through-line endpoint.
        return [
            r for r in runs
            if 2 <= len(r) <= max_line_width
        ]

    top_runs = edge_runs(black[:edge_rows])
    bottom_runs = edge_runs(black[-edge_rows:])

    if not top_runs or not bottom_runs:
        return None

    y_top = (edge_rows - 1) / 2.0
    y_bottom = height - 1 - y_top
    rows = np.arange(height)
    candidates = []

    for bottom in bottom_runs:
        for top in top_runs:
            x_bottom = float(bottom.mean())
            x_top = float(top.mean())

            slope = (x_bottom - x_top) / (y_bottom - y_top)

            # Only handle modest tilt here.
            # Strong bends remain with the existing corner logic.
            if abs(slope) > 0.8:
                continue

            xs = x_top + slope * (rows - y_top)
            supported = np.zeros(height, dtype=bool)

            # Small tolerance for image noise and tape edges.
            for offset in (-2, -1, 0, 1, 2):
                xi = np.rint(xs + offset).astype(int)
                valid = (xi >= 0) & (xi < width)
                supported[valid] |= black[rows[valid], xi[valid]]

            coverage = float(supported.mean())

            # Require a connected-looking path across almost all rows.
            if coverage < STROKE_FILL:
                continue

            midpoint = x_top + slope * (
                (height - 1) / 2.0 - y_top
            )
            candidates.append((coverage, midpoint))

    if not candidates:
        return None

    # Multiple plausible paths: keep the original fallback behavior.
    centers = [candidate[1] for candidate in candidates]
    if max(centers) - min(centers) > 0.08 * width:
        return None

    return float(max(candidates, key=lambda item: item[0])[1])

def find_junction_y(mask, height, width):
    """v10: lowest horizontal stroke (crossbar or corner) as a fraction of image height, or None."""
    r0, r1 = int(CROSS_SEARCH[0] * height), int(CROSS_SEARCH[1] * height)
    counts = (mask[r0:r1] > 0).sum(axis=1)
    rows = np.flatnonzero(counts > CROSS_ROW_FRACTION * width)
    if rows.size == 0:
        return None
    runs = [r for r in np.split(rows, np.flatnonzero(np.diff(rows) > 1) + 1) if len(r) >= CROSS_MIN_ROWS]
    if not runs:
        return None
    return (r0 + float(runs[-1].mean())) / height

def detect_line(frame, gray, green_colour):
    height, width = frame.shape[:2]
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    # v10: dark green tape must never be mistaken for black tape
    mask[cv2.dilate(green_colour, np.ones((5, 5), np.uint8)) > 0] = 0
    overlay = frame.copy()
    positions, crossing = {}, set()
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name] = None
        if contours:
            contour = max(contours, key=cv2.contourArea)
            fraction = cv2.contourArea(contour) / float(strip.size)
            m = cv2.moments(contour)
            if 0.005 < fraction < 0.80 and m["m00"] > 0:
                cx = m["m10"] / m["m00"]
                if cv2.boundingRect(contour)[2] > CROSS_BAND_WIDTH * width:   # v10
                    sx = through_line_x(strip)
                    if sx is not None:
                        cx = sx
                        crossing.add(name)
                positions[name] = (cx - width / 2.0) / (width / 2.0)
                cv2.circle(overlay, (int(cx), (y0 + y1) // 2), 4, (0, 255, 0), -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
    return positions, crossing, mask, overlay

def get_line_points(positions, width, height):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ("far", "middle", "near") if positions.get(name) is not None]

def classify_green_direction(positions, markers, width, height, junction_y):
    """Returns (direction, number of markers in the trigger zone)."""
    zone = [m for m in markers if m["y"] >= height * GREEN_TRIGGER_Y]
    if not markers:
        return "NONE", 0
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return "UNKNOWN", len(zone)
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return "UNKNOWN", len(zone)
    dx, dy = dx / length, dy / length
    sides = {"LEFT": [], "RIGHT": []}
    for marker in zone:
        # v10: rule 1.5.6 - a valid marker sits just BEFORE a junction, so a crossbar must be visible above it.
        # This rejects markers beyond the crossbar and markers already passed.
        if junction_y is None or marker["y"] <= junction_y * height:
            continue
        score = (marker["x"] - near_x) * dy - (marker["y"] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides["LEFT" if score > 0 else "RIGHT"].append(marker)
    if sides["LEFT"] and sides["RIGHT"]:
        paired = any(abs(a["y"] - b["y"]) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides["LEFT"] for b in sides["RIGHT"])
        return ("BOTH" if paired else "UNKNOWN"), len(zone)
    if sides["LEFT"]:
        return "LEFT", len(zone)
    if sides["RIGHT"]:
        return "RIGHT", len(zone)
    return "CENTER", len(zone)

def detect_finish_bands(frame):
    """Find short, broad red and bright neutral bands in the lower camera view."""
    height, width = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    y0, y1 = int(height * FINISH_ROI_TOP), int(height * FINISH_ROI_BOTTOM)
    roi = hsv[y0:y1]
    red1 = cv2.inRange(roi, np.array([0, 75, 45], np.uint8), np.array([12, 255, 255], np.uint8))
    red2 = cv2.inRange(roi, np.array([168, 75, 45], np.uint8), np.array([180, 255, 255], np.uint8))
    red = cv2.bitwise_or(red1, red2)
    silver = cv2.inRange(roi, np.array([0, 0, 135], np.uint8), np.array([180, 75, 255], np.uint8))
    # White floor can also be bright and neutral. Require a local brightness
    # change at the silver band's edge so a large uniform floor region is rejected.
    gray_roi = cv2.cvtColor(frame[y0:y1], cv2.COLOR_BGR2GRAY).astype(np.float32)
    row_mean = gray_roi.mean(axis=1)
    edge_change = np.zeros_like(row_mean)
    if len(row_mean) > 6:
        edge_change[3:-3] = np.maximum(
            np.abs(row_mean[3:-3] - row_mean[:-6]),
            np.abs(row_mean[3:-3] - row_mean[6:])
        )

    def broad_band(mask, min_coverage, require_edge=False):
        coverage = (mask > 0).mean(axis=1)
        eligible = coverage >= min_coverage
        if require_edge:
            eligible &= edge_change >= 5.0
        rows = np.flatnonzero(eligible)
        if rows.size < FINISH_ROW_MIN_PIXELS:
            return None
        runs = [r for r in np.split(rows, np.flatnonzero(np.diff(rows) > 1) + 1)
                if FINISH_ROW_MIN_PIXELS <= len(r) <= int(0.22 * height)]
        if not runs:
            return None
        run = max(runs, key=len)
        return (y0 + float(run.mean())) / height

    return (broad_band(red, FINISH_RED_MIN_COVERAGE),
            broad_band(silver, FINISH_SILVER_MIN_COVERAGE, require_edge=True))

def perceive(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    markers, green_colour, green_roi = detect_green(hsv, height)
    positions, crossing, mask, overlay = detect_line(frame, gray, green_colour)
    junction_y = find_junction_y(mask, height, width)
    direction, zone = classify_green_direction(positions, markers, width, height, junction_y)
    finish = detect_finish_bands(frame)
    return Observation(positions, direction, junction_y, zone, markers, crossing, finish), overlay, green_roi

### 3b. Synthetic self-test (no motors)
Software-in-the-loop check of perception on drawn frames. Run it on the Jetson after changing any threshold.

In [ ]:
def synth(lines=(), markers=(), thickness=10):
    img = np.full((224, 224, 3), 235, np.uint8)
    for x0, y0, x1, y1 in lines:
        cv2.line(img, (x0, y0), (x1, y1), (20, 20, 20), thickness)
    for x, y in markers:
        cv2.rectangle(img, (x - 7, y - 7), (x + 7, y + 7), (40, 170, 40), -1)
    return img

def self_test(verbose=True):
    V = (112, 0, 112, 223)
    cases = [
        ("straight line", synth([V]), lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is None),
        ("4-way: middle band not pulled sideways", synth([V, (0, 140, 223, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is not None),
        ("T with left branch: keep through-line", synth([V, (0, 140, 112, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05),
        ("unmarked left corner: still steer left", synth([(112, 223, 112, 140), (112, 140, 0, 140)]),
         lambda o: o.positions["middle"] is not None and o.positions["middle"] < -0.2),
        ("left marker before junction", synth([V, (0, 140, 223, 140)], [(94, 156)]),
         lambda o: o.direction == "LEFT"),
        ("right marker before junction", synth([V, (0, 140, 223, 140)], [(130, 156)]),
         lambda o: o.direction == "RIGHT"),
        ("dead end (both markers)", synth([V, (0, 140, 223, 140)], [(94, 156), (130, 156)]),
         lambda o: o.direction == "BOTH"),
        ("marker beyond the crossbar is ignored", synth([V, (0, 175, 223, 175)], [(94, 157)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
        ("marker with no junction is ignored", synth([V], [(94, 170)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
    ]
    passed = 0
    for name, img, check in cases:
        obs = perceive(img)[0]
        ok = bool(check(obs))
        passed += ok
        if verbose:
            pos = {k: (None if v is None else round(v, 2)) for k, v in obs.positions.items()}
            jy = None if obs.junction_y is None else round(obs.junction_y, 2)
            print("%-42s %s  dir=%-7s junction=%s %s" % (name, "PASS" if ok else "FAIL", obs.direction, jy, pos))
    print("%d/%d passed" % (passed, len(cases)))
    return passed == len(cases)

self_test()

## 4. BlackLineController
v11: in-place pivot, progress-based turn watchdog, search time derived from rotation speed, zig-zag handling.

In [ ]:
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_EXIT_MIDDLE = 0.35        # v10: middle band need only be roughly aligned; the next bend may already be visible
LOOKAHEAD_GAIN = 0.40
ZIGZAG_SPREAD = 0.60           # v10: near and far on opposite sides by this much = two bends in view
STEERING_FILTER_ALPHA = 0.55  # v11: light smoothing; lower is smoother but slower to react
TURN_PROGRESS = 0.03
TURN_STALL_TIMEOUT = 0.8       # v10: fail only if the error stops shrinking for this long
TURN_HARD_TIMEOUT = 0.5 * LINE_PIVOT_TIME_360   # v10: never rotate more than ~180 degrees
SEARCH_TIMEOUT = LINE_PIVOT_TIME_360 / 3.0      # v10: search up to ~120 degrees (v9: 0.35 s)

# 일반 검은 선 회전의 시작 조건
LINE_CENTER_LIMIT = 0.22
LINE_MIDDLE_CENTER_LIMIT = 0.35

LINE_TURN_NEAR_MIN = 0.40
LINE_TURN_MIDDLE_MIN = 0.20
LINE_TURN_CONFIRM_FRAMES = 3

# 중간 선이 안 보이는 코너용 조건
LINE_TURN_NEAR_ONLY_MIN = 0.70
LINE_TURN_NEAR_ONLY_FRAMES = 5

# 회전 확정 전 일반 조향에 사용하는 최대 오차
LINE_PENDING_ERROR_LIMIT = 0.30

class BlackLineController:
    """Follow black tape, handle sharp bends and search briefly when tape is lost."""
    def __init__(self):
        self.turn_candidate = 0
        self.turn_candidate_frames = 0
        self.turn_candidate_near_only = False
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False
        self.best_turn_error = None
        self.last_progress = None
        self.filtered_error = None  # v11: only used for ordinary line following

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = clamp(PIVOT_INNER_RATIO * speed, -MAX_SPEED, 0.0)   # v10
        return (speed, inner) if direction > 0 else (inner, speed)

    def start_turn(self, direction, now, error=1.0):
        self.turn_started = now
        self.turn_direction = direction
        self.best_turn_error = abs(error)
        self.last_progress = now

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near, middle = positions.get("near"), positions.get("middle")
        visible = [(name, value) for name, value in positions.items() if value is not None]
        
        if not visible:
            self.turn_candidate = 0
            self.turn_candidate_frames = 0
            self.turn_candidate_near_only = False

            if self.lost_started is None:
                self.lost_started = now

            if not self.last_direction:
                return self.stop("line lost without a known turn direction")

            if now - self.lost_started >= SEARCH_TIMEOUT:
                return self.stop("line not reacquired")

            if self.turn_started is None:
                self.start_turn(self.last_direction, now)

            if now - self.turn_started >= TURN_HARD_TIMEOUT:
                return self.stop("turn timed out")

            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"
        

        returning = self.lost_started is not None
        self.lost_started = None
        
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}

        farthest = positions.get("far")
        if farthest is None:
            farthest = middle

        # 가까운 선과 중간 선이 중앙에 있으면
        # 먼 선의 옆 가지에 끌려가지 않도록 far를 제외.
        near_aligned = (
            near is not None
            and abs(near) < LINE_CENTER_LIMIT
        )
        middle_aligned = (
            middle is None
            or abs(middle) < LINE_MIDDLE_CENTER_LIMIT
        )

        protect_forward = (
            self.turn_started is None
            and near_aligned
            and middle_aligned
        )

        zigzag = (
            near is not None
            and farthest is not None
            and near * farthest < 0
            and abs(farthest - near) >= ZIGZAG_SPREAD
        )

        if protect_forward or zigzag:
            used = [
                (name, value)
                for name, value in visible
                if name != "far"
            ]
        else:
            used = visible

        error = (
            sum(weights[name] * value for name, value in used)
            / sum(weights[name] for name, _ in used)
        )

        # 먼 선을 향한 선행 조향도 같은 조건에서 제외.
        if (
            not protect_forward
            and not zigzag
            and near is not None
            and farthest is not None
        ):
            error += LOOKAHEAD_GAIN * (farthest - near)

        error = clamp(error, -1.0, 1.0)
        
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            if returning:   # line reappeared after a search: restart progress tracking from here
                self.best_turn_error, self.last_progress = abs(error), now
            if near is not None and abs(near) < TURN_EXIT and (middle is None or abs(middle) < TURN_EXIT_MIDDLE):
                self.turn_started = None
                self.turn_direction = 0
            else:
                if abs(error) < self.best_turn_error - TURN_PROGRESS:
                    self.best_turn_error, self.last_progress = abs(error), now
                if now - self.last_progress >= TURN_STALL_TIMEOUT:
                    return self.stop("turn made no progress")
                if now - self.turn_started >= TURN_HARD_TIMEOUT:
                    return self.stop("turn timed out")

        if self.turn_started is None:
                    
            candidate = 0
            near_only = False
            required_frames = LINE_TURN_CONFIRM_FRAMES

            if abs(error) >= TURN_ENTER:
                error_direction = 1 if error > 0 else -1

                both_support_turn = (
                    near is not None
                    and middle is not None
                    and abs(near) >= LINE_TURN_NEAR_MIN
                    and abs(middle) >= LINE_TURN_MIDDLE_MIN
                    and near * middle > 0
                    and near * error_direction > 0
                )

                near_supports_turn_alone = (
                    near is not None
                    and middle is None
                    and abs(near) >= LINE_TURN_NEAR_ONLY_MIN
                    and near * error_direction > 0
                )

                if both_support_turn:
                    candidate = error_direction
                elif near_supports_turn_alone:
                    candidate = error_direction
                    near_only = True
                    required_frames = LINE_TURN_NEAR_ONLY_FRAMES

            if candidate == 0:
                self.turn_candidate = 0
                self.turn_candidate_frames = 0
                self.turn_candidate_near_only = False
            else:
                same_evidence = (
                    candidate == self.turn_candidate
                    and near_only == self.turn_candidate_near_only
                )

                if same_evidence:
                    self.turn_candidate_frames += 1
                else:
                    self.turn_candidate = candidate
                    self.turn_candidate_frames = 1
                    self.turn_candidate_near_only = near_only

                if self.turn_candidate_frames >= required_frames:
                    self.start_turn(candidate, now, error)
                    self.turn_candidate = 0
                    self.turn_candidate_frames = 0
                    self.turn_candidate_near_only = False

            if (
                self.turn_started is None
                and abs(error) >= TURN_ENTER
            ):
                reference = near if near is not None else 0.0
                error = clamp(
                    reference,
                    -LINE_PENDING_ERROR_LIMIT,
                    LINE_PENDING_ERROR_LIMIT,
                )
                self.filtered_error = error
                
        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        base = MIN_DRIVE_SPEED if zigzag else max(base, MIN_DRIVE_SPEED)   # v10: slow, but never below stall
        # v11: smooth normal steering only. Sharp turns and zig-zags keep the
        # original immediate response so their proven behavior is preserved.
        steering_error = error
        if not zigzag and abs(error) < TURN_ENTER:
            if self.filtered_error is None:
                self.filtered_error = error
            else:
                self.filtered_error = (STEERING_FILTER_ALPHA * error
                                       + (1.0 - STEERING_FILTER_ALPHA) * self.filtered_error)
            steering_error = self.filtered_error
        else:
            self.filtered_error = error
        correction = KP * steering_error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, ("FOLLOW_SLOW" if zigzag else "FOLLOW")

## 5. GapCrossingController (unchanged from v9)

In [ ]:
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 2.3
GAP_STEER_LIMIT = 0.015
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCrossingController:
    """Own gap evidence/crossing; delegate ordinary driving to BlackLineController."""
    def __init__(self):
        self.black_line = BlackLineController()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.black_line.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near, middle = positions.get("near"), positions.get("middle")
        values = [v for v in positions.values() if v is not None]

        if self.gap_started is not None:
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.black_line.stop("gap crossing timed out")
            self.confirm_frames = self.confirm_frames + 1 if (near is not None and middle is not None) else 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.black_line.lost_started = None
            self.black_line.last_direction = 0

        if not values:
            recent = self.straight_seen is not None and now - self.straight_seen <= GAP_APPROACH_MAX_AGE
            if self.black_line.turn_started is None and recent and self.straight_frames >= GAP_APPROACH_FRAMES:
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return self.black_line.step(positions, now)

        left, right, state = self.black_line.step(positions, now)
        aligned = max(abs(v) for v in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        return left, right, state

## 6. GreenMarkerTurnController
v10 sequence (preserved): vote → STOP (may upgrade to BOTH) → ADVANCE until the crossbar passes under the camera,
then CAMERA_BLIND_TIME → TURN (camera confirms reacquisition) → DONE. A dead end (BOTH) skips the advance.

In [ ]:
GREEN_VOTE_WINDOW = 5          # v10: k-of-n confirmation
GREEN_VOTES_NEEDED = 3
GREEN_BOTH_UPGRADE = 2         # v10: BOTH frames seen while stopped that turn a LEFT/RIGHT into a U-turn
GREEN_STOP_TIME = 0.20
GREEN_ADVANCE_SPEED = BASE_SPEED
GREEN_ADVANCE_STEER = 0.05
JUNCTION_NEAR_Y = 0.75         # crossbar must have come this low before its disappearance counts
JUNCTION_EXIT_Y = 0.90         # ...or reached this row
JUNCTION_JUMP = 0.20           # a stroke this much higher is a different (farther) junction
GREEN_NO_JUNCTION_WAIT = 0.30  # crossbar never seen while advancing: assume already under the camera
GREEN_ADVANCE_TIMEOUT = 2.0
GREEN_TURN_MIN_TIME = 0.6 * QUARTER_TURN_TIME
GREEN_TURN_TIMEOUT = 2.5 * QUARTER_TURN_TIME
UTURN_MIN_TIME = 0.75 * 2 * QUARTER_TURN_TIME
UTURN_TIMEOUT = 2.0 * 2 * QUARTER_TURN_TIME
GREEN_REACQUIRE_FRAMES = 3
GREEN_COOLDOWN = 0.4
GREEN_CLEAR_FRAMES = 0         # v10: 0 = cooldown only; the junction-above rule already blocks the old marker
GREEN_TURN_SLOW_SPEED = 0.10   # CALIBRATE: lowest speed that still reliably rotates in place
GREEN_SLOW_ZONE = 0.6          # line within this offset in near or middle -> slow down
FEEDFORWARD_FRACTION = 0.75    # fraction of the expected angle done fast, open-loop

def line_centered(positions):
    return all(positions.get(n) is not None and abs(positions[n]) < TURN_EXIT for n in ("near", "middle"))

# 우선 기존에 움직임이 확인된 속도로 시작.
# 0.10에서도 안정적으로 전진하는 것을 확인했다면 0.10으로 조정.
GREEN_APPROACH_SPEED = 0.12

# 기존 advance()가 이 값을 사용함.
GREEN_ADVANCE_SPEED = GREEN_APPROACH_SPEED

# 접근 중 안쪽 바퀴가 너무 느려지는 것을 줄임.
GREEN_ADVANCE_STEER = 0.02

# 접근하면서 LEFT/RIGHT -> BOTH 추가 확인을 허용하는 시간.
GREEN_RECHECK_TIME = 0.20

# 전진 명령에서 제자리 회전 명령으로 전환하는 시간.
GREEN_TURN_RAMP_TIME = 0.12

class GreenMarkerTurnController:
    """Confirm green instructions and execute them; return None when not in control."""
    def __init__(self):
        self.state = "FOLLOW"
        self.state_started = None
        self.direction = None
        self.votes = deque(maxlen=GREEN_VOTE_WINDOW)
        self.both_votes = 0
        self.armed = True
        self.clear_frames = 0
        self.cooldown_until = 0.0
        self.departed = False
        self.reacquire_frames = 0
        self.stopped = False
        self.reason = ""

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, "STOP: " + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    def turn(self, positions, elapsed):
        expected = (2 if self.direction == "BOTH" else 1) * QUARTER_TURN_TIME
        near, middle = positions.get("near"), positions.get("middle")
        line_close = any(v is not None and abs(v) < GREEN_SLOW_ZONE for v in (near, middle))
        slow = elapsed >= FEEDFORWARD_FRACTION * expected and line_close
        speed = clamp(GREEN_TURN_SLOW_SPEED if slow else GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        suffix = "_SLOW" if slow else ""
        if self.direction == "RIGHT":
            return speed, -speed, "GREEN_TURN_RIGHT" + suffix
        label = "GREEN_U_TURN" if self.direction == "BOTH" else "GREEN_TURN_LEFT"
        return -speed, speed, label + suffix

    def confirmed(self):
        counts = {d: self.votes.count(d) for d in ("LEFT", "RIGHT", "BOTH")}
        best = max(counts, key=counts.get)
        return best if counts[best] >= GREEN_VOTES_NEEDED else None

    def begin_advance(self, now):
        self.junction_seen = False
        self.last_junction_y = None
        self.passed_frames = 0
        self.post_started = None
        self.enter("GREEN_ADVANCE", now)

    def begin_turn(self, now):
        self.departed = False
        self.reacquire_frames = 0
        self.enter("GREEN_TURN", now)

    def advance(self, obs, now):
        """v10: closed-loop approach. Returns a command, or None when the axle is over the junction."""
        elapsed = now - self.state_started
        if elapsed >= GREEN_ADVANCE_TIMEOUT:
            return self.stop("junction not passed during advance")
        if self.post_started is None:
            jy = obs.junction_y
            if jy is not None and (self.last_junction_y is None or jy >= self.last_junction_y - JUNCTION_JUMP):
                self.junction_seen = True
                self.last_junction_y = jy
                under = jy >= JUNCTION_EXIT_Y
            elif self.junction_seen:
                under = self.last_junction_y >= JUNCTION_NEAR_Y
            else:
                under = elapsed >= GREEN_NO_JUNCTION_WAIT
            self.passed_frames = self.passed_frames + 1 if under else 0
            if self.passed_frames >= 2:
                self.post_started = now
        elif now - self.post_started >= CAMERA_BLIND_TIME:
            return None
        near = obs.positions.get("near")
        steer = clamp(KP * near, -GREEN_ADVANCE_STEER, GREEN_ADVANCE_STEER) if near is not None else 0.0
        speed = clamp(GREEN_ADVANCE_SPEED, 0.0, MAX_SPEED)
        label = "GREEN_ADVANCE_BLIND" if self.post_started is not None else "GREEN_ADVANCE"
        return clamp(speed + steer, 0.0, MAX_SPEED), clamp(speed - steer, 0.0, MAX_SPEED), label

    def step(self, obs, now, can_start=True):
        if self.stopped:
            return self.stop(self.reason)

        positions = obs.positions
        direction = obs.direction

        # 1. 평상시: 기존 3-of-5 방식으로 마커 방향 확정
        if self.state == "FOLLOW":
            if not self.armed:
                self.clear_frames = (
                    self.clear_frames + 1
                    if direction not in ("LEFT", "RIGHT", "BOTH")
                    else 0
                )
                if (
                    now >= self.cooldown_until
                    and self.clear_frames >= GREEN_CLEAR_FRAMES
                ):
                    self.armed = True

            if not (self.armed and can_start):
                self.votes.clear()
                return None

            self.votes.append(direction)
            decided = self.confirmed()

            if decided is None:
                return None

            self.direction = decided
            self.armed = False
            self.votes.clear()
            self.both_votes = 0
            self.recheck_until = now + GREEN_RECHECK_TIME

            # Runner가 기록한 직전 모터 명령.
            # preview 또는 초기 호출에서는 기본 전진 속도를 사용.
            self.approach_command = getattr(
                self,
                "last_motor_command",
                (BASE_SPEED, BASE_SPEED),
            )

            if decided == "BOTH":
                # 유턴: 교차로로 더 접근하지 않고 회전 전환 시작.
                self.ramp_from = self.approach_command
                self.begin_turn(now)
            else:
                # 좌/우회전: 멈추지 않고 교차로 접근 시작.
                self.begin_advance(now)

        # 2. 저속 접근: 기존 교차로 통과 판단 유지
        if self.state == "GREEN_ADVANCE":
            if now <= self.recheck_until and direction == "BOTH":
                self.both_votes += 1

            if self.both_votes >= GREEN_BOTH_UPGRADE:
                # 접근 중 양쪽 마커가 확인되면 유턴으로 변경.
                self.direction = "BOTH"
                self.ramp_from = self.approach_command
                self.begin_turn(now)
            else:
                command = self.advance(obs, now)

                if command is not None:
                    # advance()의 timeout 정지도 그대로 반환.
                    if not self.stopped:
                        self.approach_command = command[:2]
                    return command

                # 교차로 접근 완료: 마지막 전진 명령에서 회전으로 연결.
                self.ramp_from = self.approach_command
                self.begin_turn(now)

        # 3. 회전 진입 + 검은 선 재탐색
        if self.state == "GREEN_TURN":
            elapsed = now - self.state_started

            centered = line_centered(positions)
            if not centered:
                self.departed = True

            # 전진 -> 제자리 회전으로 부드럽게 전환.
            # 이 구간에서는 선 재인식으로 회전을 종료하지 않음.
            if elapsed < GREEN_TURN_RAMP_TIME:
                alpha = clamp(
                    elapsed / GREEN_TURN_RAMP_TIME, 0.0, 1.0
                )
                # 시작/끝의 속도 변화가 완만한 보간.
                blend = alpha * alpha * (3.0 - 2.0 * alpha)

                target_left, target_right, _ = self.turn(
                    positions, 0.0
                )
                start_left, start_right = self.ramp_from

                left = start_left + blend * (
                    target_left - start_left
                )
                right = start_right + blend * (
                    target_right - start_right
                )

                return left, right, "GREEN_TURN_RAMP"

            # 최소 회전 시간은 전환 구간이 끝난 뒤부터 계산.
            turn_elapsed = elapsed - GREEN_TURN_RAMP_TIME

            limit = (
                UTURN_TIMEOUT
                if self.direction == "BOTH"
                else GREEN_TURN_TIMEOUT
            )
            minimum = (
                UTURN_MIN_TIME
                if self.direction == "BOTH"
                else GREEN_TURN_MIN_TIME
            )

            if turn_elapsed >= limit:
                return self.stop("green turn timed out")

            if (
                self.departed
                and turn_elapsed >= minimum
                and centered
            ):
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0

            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:
                return self.turn(positions, turn_elapsed)

            # 완료 신호. 실제 주행 명령은 coordinator가
            # 같은 프레임에서 새 라인 추종 명령으로 교체함.
            self.enter("FOLLOW", now)
            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0
            self.votes.clear()

            return 0.0, 0.0, "GREEN_DONE"

        return None

## 7. NavigationCoordinator
v11: takes an `Observation`. A green maneuver may now start during a black-line TURN **if a junction is
visible** (that "turn" was caused by the crossbar). It still never interrupts a gap crossing.

In [ ]:
class NavigationCoordinator:
    """Single decision point: green maneuver > gap crossing > black-line control."""
    def __init__(self):
        self.green_turn = GreenMarkerTurnController()
        self.gap = GapCrossingController()
        self.stopped = False
        self.reason = ""
        self.active_controller = "BLACK_LINE"
        self.finish_red_seen_at = None  # v11: remember red before accepting silver

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        self.active_controller = "STOPPED"
        return 0.0, 0.0, "STOP: " + reason

    def step(self, obs, now):
        if self.stopped:
            return self.stop(self.reason)
        # v11: only finish after red has been observed and silver follows in time.
        red_y, silver_y = obs.finish
        if red_y is not None:
            self.finish_red_seen_at = now
        if self.finish_red_seen_at is not None:
            if now - self.finish_red_seen_at > FINISH_SEQUENCE_TIMEOUT:
                self.finish_red_seen_at = None
            elif silver_y is not None and (red_y is None or silver_y >= red_y - 0.05):
                self.stopped = True
                self.reason = "FINISH_LINE_DETECTED (red band followed by silver foil)"
                self.active_controller = "FINISHED"
                return 0.0, 0.0, self.reason
        line_turning = self.gap.black_line.turn_started is not None
        can_start_green = (self.gap.gap_started is None and
                           (not line_turning or obs.junction_y is not None))   # v10
        command = self.green_turn.step(obs, now, can_start=can_start_green)
        if command is not None:
            self.active_controller = "GREEN_TURN"
            if self.green_turn.stopped:
                self.stopped = True
                self.reason = self.green_turn.reason
                self.active_controller = "STOPPED"
            
            elif command[2] == "GREEN_DONE":
                self.gap = GapCrossingController()
                self.active_controller = "BLACK_LINE"

                # GREEN_DONE의 (0, 0)을 모터에 보내지 않고
                # 같은 프레임에서 바로 라인 추종으로 연결.
                command = self.gap.step(obs.positions, now)

                if self.gap.black_line.stopped:
                    return self.stop(command[2])

            return command
        
        command = self.gap.step(obs.positions, now)
        self.active_controller = "GAP_CROSSING" if self.gap.gap_started is not None else "BLACK_LINE"
        if self.gap.black_line.stopped:
            return self.stop(command[2])
        return command

## 8. Runner + logging
The control loop does no file I/O: each frame appends one tuple to a bounded ring buffer, and the CSV is
written after the motors have stopped. The summary reports loop rate, processing time and frame age -
use these numbers in the report's timing discussion.

In [ ]:
LOG_FIELDS = ("t", "frame_age", "proc_ms", "owner", "state", "marker", "near", "middle", "far",
              "junction_y", "zone_markers", "left", "right")
LOG = deque(maxlen=6000)   # ~5 min at 20 Hz; memory stays bounded

def summarize_log(save=True):
    rows = list(LOG)
    if len(rows) < 2:
        print("No log data.")
        return
    t = np.array([r[0] for r in rows]); proc = np.array([r[2] for r in rows]); age = np.array([r[1] for r in rows])
    periods = np.diff(t)
    print("frames %d | loop %.1f Hz (period mean %.0f ms, max %.0f ms) | processing mean %.1f ms, p95 %.1f ms, max %.1f ms | frame age max %.0f ms"
          % (len(rows), 1.0 / periods.mean(), 1000 * periods.mean(), 1000 * periods.max(),
             proc.mean(), np.percentile(proc, 95), proc.max(), 1000 * age.max()))
    previous = None
    for r in rows:
        key = (r[3], r[4].split(":")[0])
        if key != previous:
            print("  t=%6.2f  %-12s %-22s marker=%-7s junction=%s" % (r[0], r[3], r[4], r[5],
                  "-" if r[9] is None else "%.2f" % r[9]))
            previous = key
    if save:
        name = "run_%d.csv" % int(time.time())
        with open(name, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(LOG_FIELDS)
            writer.writerows(rows)
        print("Saved", name)

In [ ]:
overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=336)
status_view = widgets.Label(value="Ready for stationary preview")
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_v10(seconds=10, drive=False, show=True):
    robot.stop()
    controller = NavigationCoordinator()
    LOG.clear()
    start = time.monotonic()
    deadline = start + seconds
    previous_stamp = None
    next_display = 0.0
    display_period = 0.3 if drive else 0.1   # v10: JPEG encoding is non-essential work; do less of it while driving
    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]
            if sample is None:
                print("Stopped: no camera frame")
                break
            frame, captured_at = sample
            now = time.monotonic()
            if now - captured_at > FRAME_TIMEOUT:
                print("Stopped: stale camera frame")
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            t0 = time.monotonic()
            obs, overlay, green_mask = perceive(frame)
            now = time.monotonic()
            proc_ms = 1000.0 * (now - t0)
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print("Stopped: deadline or slow image processing")
                break
            
            left, right, state = controller.step(obs, now)

            # 다음 프레임의 회전 전환에 사용할 직전 출력.
            controller.green_turn.last_motor_command = (left, right)

            if drive:
                robot.set_motors(left, right)
                
            p = obs.positions
            LOG.append((now - start, now - captured_at, proc_ms, controller.active_controller, state,
                        obs.direction, p.get("near"), p.get("middle"), p.get("far"),
                        obs.junction_y, obs.zone_markers, left, right))
            if show and now >= next_display:
                height, width = frame.shape[:2]
                points = get_line_points(p, width, height)
                for a, b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int, a)), tuple(map(int, b)), (255, 0, 255), 2)
                for marker in obs.markers:
                    x, y, w, h = marker["box"]
                    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
                if obs.junction_y is not None:
                    jy = int(obs.junction_y * height)
                    cv2.line(overlay, (0, jy), (width - 1, jy), (0, 0, 255), 1)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = "%s | %s | marker=%s | L=%.2f R=%.2f | owner=%s" % (
                    "DRIVE" if drive else "PREVIEW", state, obs.direction, left, right, controller.active_controller)
                next_display = now + display_period
            if controller.stopped:
                print("FINISH:" if controller.active_controller == "FINISHED" else "FAILURE:", state)
                break
            time.sleep(0.005)
    finally:
        robot.stop()
        print("Run ended; motors stopped.")
        summarize_log()

def run_preview(seconds=20):
    run_v10(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v10(seconds=seconds, drive=True)

## 9. Test order
1. `self_test()` (cell 3b). 2. `spin_test()` → set `ROTATION_TIME_360`, re-run cell 2 and the controller cells.
3. Measure `CAMERA_BLIND_TIME`: tape across the floor, drive straight, note when it leaves the image and when it is under the axle.
4. `run_preview()` pushing the robot by hand over a marked junction: watch for the red junction line and `marker=LEFT`.
5. Floor: single left / right / both marker, then the back-to-back marker section, then tight bends and gaps.
After every run read the printed transition list — it shows which state ended the run.

In [68]:
run_preview(seconds=40)

Run ended; motors stopped.
frames 1201 | loop 30.0 Hz (period mean 33 ms, max 45 ms) | processing mean 5.8 ms, p95 6.9 ms, max 13.5 ms | frame age max 37 ms
  t=  0.01  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
Saved run_1793616868.csv


In [71]:
# Explicitly uncomment to move the robot:
run_robot(seconds=40)

Run ended; motors stopped.
frames 274 | loop 28.9 Hz (period mean 35 ms, max 87 ms) | processing mean 6.3 ms, p95 10.1 ms, max 12.3 ms | frame age max 39 ms
  t=  0.01  BLACK_LINE   FOLLOW                 marker=NONE    junction=0.71
  t=  0.14  BLACK_LINE   FOLLOW_SLOW            marker=NONE    junction=0.72
  t=  0.43  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  0.51  BLACK_LINE   TURN                   marker=NONE    junction=0.92
  t=  0.81  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  1.77  BLACK_LINE   FOLLOW_SLOW            marker=CENTER  junction=-
  t=  1.80  BLACK_LINE   FOLLOW                 marker=CENTER  junction=0.63
  t=  2.15  BLACK_LINE   FOLLOW_SLOW            marker=CENTER  junction=0.90
  t=  2.25  BLACK_LINE   FOLLOW                 marker=CENTER  junction=0.91
  t=  3.08  BLACK_LINE   TURN                   marker=NONE    junction=-
  t=  4.08  BLACK_LINE   FOLLOW                 marker=NONE    junction=0.88
  t=  5.

KeyboardInterrupt: 

## 10. Cleanup

In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()